In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB,BernoulliNB,ComplementNB,MultinomialNB
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier



from sklearn.metrics import accuracy_score

In [2]:
np.random.seed(42)

In [3]:
dataset=pd.read_csv('tweet_emotions.csv')

In [4]:
dataset.isna().sum()

tweet_id     0
sentiment    0
content      0
dtype: int64

In [5]:
dataset['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [6]:
labelled_dataset=dataset[dataset['sentiment'] != 'empty']

In [7]:
# extracting 1000 records of each dataset
neutral_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'neutral'][:500]
worry_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'worry'][:500]
happiness_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'happiness'][:500]
sadness_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'sadness'][:500]
love_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'love'][:500]
surprise_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'surprise'][:500]
fun_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'fun'][:500]
relief_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'relief'][:500]
hate_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'hate'][:500]
enthusiasm_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'enthusiasm'][:500]
boredom_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'boredom'][:]
anger_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'anger'][:]

# combining these datasets together
final_training_dataset = pd.concat([neutral_dataset , worry_dataset , happiness_dataset , sadness_dataset , love_dataset , surprise_dataset , 
                                     fun_dataset , relief_dataset , hate_dataset , enthusiasm_dataset , boredom_dataset , anger_dataset],axis = 0)

In [10]:
corpus=[]
lemmatizer=WordNetLemmatizer()
content=final_training_dataset['content']
for document in content:
    tokenized_document = word_tokenize(document)
    filtered_document = [word for word in tokenized_document if word.lower() not in stopwords.words('english') and word.lower() not in string.punctuation and word.lower() not in string.digits] 
    lemmatized_document = [lemmatizer.lemmatize(document) for document in filtered_document]
    corpus.append(lemmatized_document)

In [11]:
final_training_dataset['processed_corpus'] = corpus

In [12]:
final_training_dataset.to_csv('processed_dataset.csv')

In [13]:
corpus = [' '.join(document) for document in corpus]

In [14]:
len(corpus)

5289

In [15]:
final_training_dataset

,tweet_id,sentiment,content,processed_corpus
4,1956968416,neutral,@dannycastillo We want to trade with someone w...,"[dannycastillo, want, trade, someone, Houston,..."
10,1956969456,neutral,cant fall asleep,"[cant, fall, asleep]"
22,1956972116,neutral,No Topic Maps talks at the Balisage Markup Con...,"[Topic, Maps, talk, Balisage, Markup, Conferen..."
31,1956975441,neutral,@cynthia_123 i cant sleep,"[cynthia_123, cant, sleep]"
32,1956975860,neutral,I missed the bl***y bus!!!!!!!!,"[missed, bl, bus]"
...,...,...,...,...
34762,1752943449,anger,my gawwddd ! 6 headshotss inna row? im on fyaa...,"[gawwddd, headshotss, inna, row, im, fyaaahhh]"
35160,1753032343,anger,I'm way to sleepy.. Ill watch my shows lata..G...,"['m, way, sleepy, .., Ill, watch, show, lata, ..."
35913,1753199183,anger,@NerdIndian Take that back. I am insulted.,"[NerdIndian, Take, back, insulted]"
36211,1753257239,anger,@anieszkaa haha i did a ltiitle bit yesterday ...,"[anieszkaa, haha, ltiitle, bit, yesterday, ive..."


In [16]:
vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(corpus[:])
# x = x.astype('float16')
x = x.toarray()

In [17]:
x.shape()

TypeError: 'tuple' object is not callable

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# Split data
y = labelled_dataset['sentiment'][:len(x)]
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2)

catboost = CatBoostClassifier(iterations=5000)
bnb = BernoulliNB()
mnb = MultinomialNB()
ada = AdaBoostClassifier(n_estimators=5000)
# Define models
models = {
    # "GaussianNB": GaussianNB(),
    "BernoulliNB": bnb,
    "MultinomialNB": mnb,
    # "AdaBoostClassifier": ada,
    # "ComplementNB": ComplementNB(),
    # "CatBoostClassifier":catboost,
    # "XGBClassifier":XGBClassifier(),
    # "MLPClassifier":MLPClassifier()
}

# Train, predict, and calculate accuracy
accuracies = {}
for name, model in models.items():
    print(name)
    model.fit(xtrain, ytrain)
    ypred = model.predict(xtest)
    accuracies[name] = accuracy_score(ytest, ypred)
    print(accuracy_score(ytest, ypred))

BernoulliNB
0.32325141776937616
MultinomialNB
0.32892249527410206
AdaBoostClassifier


d:\IDEs\anaconda\envs\assignment-env\lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


0.32608695652173914


In [ ]:
import pickle
pickle.dump(bnb , open('bnb.pkl','wb'))
pickle.dump(mnb , open('mnb.pkl','wb'))
pickle.dump(ada , open('ada.pkl','wb'))
pickle.dump(vectorizer,open('vectorizer.pkl','wb'))

In [ ]:
content = 'Last night, I had a very strange dream. I saw that two of my friends and I were going to attend my cousin’s wedding reception. On our way a rather strange-looking man intercepted us. Despite our protests, he insisted on narrating to us his tale of resentment. He looked unnaturally old with skinny limbs and glittering eyes, and the tale he narrated seemed too surreal to be true. He said he used to be a mariner who shot an innocent albatross who was guiding their ship out of the perilous ice at sea. This act enraged a powerful spirit who used to love the bird. His actions resulted in the death of his crewmates. Also, he was doomed to feel a great deal of agony which was relieved only when he narrated the tale to somebody. The dream taught me a valuable lesson: Never hurt the hand which helps you.'
corpus=[]
lemmatizer=WordNetLemmatizer()
content=final_training_dataset['content']
for document in content:
    tokenized_document = word_tokenize(document)
    filtered_document = [word for word in tokenized_document if word.lower() not in stopwords.words('english') and word.lower() not in string.punctuation and word.lower() not in string.digits] 
    lemmatized_document = [lemmatizer.lemmatize(document) for document in filtered_document]
    corpus.append(lemmatized_document)
corpus = [' '.join(document) for document in corpus]
x = vectorizer.transform(corpus[:])
# x = x.astype('float16')
x = x.toarray()

In [ ]:
print(mnb.predict(x))

['worry' 'sadness' 'worry' ... 'worry' 'worry' 'worry']
